# LSTM ile 30 gün sonrası baraj doluluk tahmini (Beyza)
**Amaç:** XGBoost ile *aynı problemi* (30 gün sonraki toplam baraj doluluğu), *aynı veriyle* ve *aynı test yılıyla* (2023) çözüp
modelleri tek bir tabloda adil şekilde karşılaştırmak.

**Neden ilçe tüketimi yerine baraj doluluğu?** İlçe tüketim verisi yıllıktır; günlüğe çevrilince her gün aynı değeri alır.
Böyle düz bir seride "yarını tahmin et" demek, dünkü değeri kopyalamakla aynıdır; model bir şey öğrenmeden mükemmel görünür.
Baraj doluluğu ise gerçekten günlük ölçülür ve yağışa, tüketime, mevsime bağlı olarak değişir.

In [ ]:
import sys; sys.path.append("..")
import os, random
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from src.veri import birlesik_veri
from src.degerlendir import bol, metrikler, UFUK
from src.ozellikler import model_verisi, modeller

# Her çalıştırmada aynı sonucu almak için rastgelelik sabitleniyor
os.environ["PYTHONHASHSEED"] = "0"; random.seed(0); np.random.seed(0); tf.random.set_seed(0)

df = birlesik_veri("../data/raw/")
print(df.shape, df.index.min().date(), df.index.max().date())

## 1. LSTM için veri hazırlama
- **Girdi:** her gün için son **90 günün** dizisi: doluluk, yağış, tüketim ve mevsim (sin/cos). LSTM bu diziyi sırayla okur.
- **Hedef:** 30 gün sonraki doluluk − bugünkü doluluk (XGBoost ile aynı). Tahmin bugünkü doluluğa eklenir.
- **Ölçekleme:** sinir ağları büyük sayılarda zorlanır, değerler 0-1 aralığına getirilir.
  Ölçekleyici **sadece eğitim verisiyle** ayarlanır; test verisini önceden "görmemek" için (veri sızıntısı olmasın).

In [ ]:
PENCERE = 90
oz = pd.DataFrame({
    "doluluk": df["toplam"], "yagis": df["yagis"], "tuketim": df["tuketim"] / 1e6,
    "mevsim_sin": np.sin(2 * np.pi * df.index.dayofyear / 365.25),
    "mevsim_cos": np.cos(2 * np.pi * df.index.dayofyear / 365.25),
})
hedef = df["toplam"].shift(-UFUK) - df["toplam"]

# Geçerli günler: önünde 90 günlük geçmiş, arkasında 30 günlük gelecek olanlar
gunler = pd.DataFrame({"doluluk": df["toplam"], "hedef": hedef}).iloc[PENCERE - 1:].dropna()
p = bol(gunler)   # src/degerlendir.py: eğitim 2011-2020, doğrulama 2021-2022, test 2023
print({k: len(v) for k, v in p.items()})

olcek_x = MinMaxScaler().fit(oz.loc[:p["egitim_tum"].index[-1]])
olcek_y = StandardScaler().fit(p["egitim_tum"][["hedef"]])
oz_olcekli = pd.DataFrame(olcek_x.transform(oz), index=oz.index, columns=oz.columns)
konum = {t: i for i, t in enumerate(oz.index)}

def diziler(parca):
    X = np.stack([oz_olcekli.values[konum[t] - PENCERE + 1: konum[t] + 1] for t in parca.index])
    y = olcek_y.transform(parca[["hedef"]]).ravel()
    return X, y

X_eg, y_eg = diziler(p["egitim"]); X_dog, y_dog = diziler(p["dogrulama"])
X_tum, y_tum = diziler(p["egitim_tum"]); X_test, _ = diziler(p["test"])
print("Girdi boyutu:", X_tum.shape, "(örnek, gün, öznitelik)")

## 2. Model
- **LSTM(16):** 90 günlük diziyi okuyup özetleyen katman (veri az olduğu için küçük tutuldu).
- **Dropout(0.2):** ezberlemeyi (overfitting) azaltmak için eğitimde nöronların %20'sini rastgele kapatır.
- **Dense(1):** tek sayı çıktı (30 günlük değişim).
- **Erken durdurma:** 2011-2020 ile eğitip 2021-2022 doğrulama hatası artık düşmeyince duruyoruz; bulunan epoch sayısıyla
  son modeli 2011-2022'nin tamamıyla eğitiyoruz (XGBoost'taki ayar yöntemiyle aynı mantık).
- **5 tohumlu ortalama:** sinir ağlarının sonucu başlangıçtaki rastgele ağırlıklara göre belirgin şekilde değişiyor
  (denemelerimizde tek bir LSTM'in test hatası 4 ile 7 puan arasında oynadı). Bu yüzden 5 farklı tohumla 5 model eğitip
  tahminlerinin ortalamasını alıyoruz ve tek tek modellerin hatasını da raporluyoruz. Böylece sonuç şansa bağlı olmuyor.

In [ ]:
def model_kur():
    m = tf.keras.Sequential([
        tf.keras.Input(shape=(PENCERE, oz.shape[1])),
        tf.keras.layers.LSTM(16),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(1),
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(3e-4), loss="mse")
    return m

TOHUMLAR = [0, 1, 2, 3, 4]
test = p["test"]
gercek = test["doluluk"] + test["hedef"]
tek_tahminler, tek_sonuc, egriler = [], [], []
for tohum in TOHUMLAR:
    tf.random.set_seed(tohum); np.random.seed(tohum)
    m = model_kur()
    g = m.fit(X_eg, y_eg, validation_data=(X_dog, y_dog), epochs=60, batch_size=64, verbose=0,
              callbacks=[tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)])
    epoch = int(np.argmin(g.history["val_loss"])) + 1
    egriler.append(g.history)
    tf.random.set_seed(tohum); np.random.seed(tohum)
    son = model_kur()
    son.fit(X_tum, y_tum, epochs=epoch, batch_size=64, verbose=0)
    d = olcek_y.inverse_transform(son.predict(X_test, verbose=0)).ravel()
    tek_tahminler.append(d)
    tek_sonuc.append({"tohum": tohum, "epoch": epoch, **metrikler(gercek, test["doluluk"] + d)})
    print(f"tohum {tohum}: {epoch} epoch, test MAE {tek_sonuc[-1]['MAE']:.2f}")

lstm_degisim = np.mean(tek_tahminler, axis=0)
tek_sonuc = pd.DataFrame(tek_sonuc)
print(f"Tek modellerin MAE'si: ortalama {tek_sonuc.MAE.mean():.2f} ± {tek_sonuc.MAE.std():.2f}")

In [ ]:
plt.figure(figsize=(7, 3))
for i, h in enumerate(egriler):
    plt.plot(h["val_loss"], alpha=0.7, label=f"tohum {TOHUMLAR[i]}")
plt.xlabel("epoch"); plt.ylabel("doğrulama hatası (MSE)"); plt.title("Öğrenme eğrileri (2021-2022 doğrulama)")
plt.legend(fontsize=7); plt.show()

## 3. Karşılaştırma (2023 testi)
Aynı test günlerinde: iki basit taban, Doğrusal Regresyon, XGBoost (Sinem) ve LSTM.
**Taban neden önemli?** "Doluluk 30 gün sonra da aynı kalır" gibi hiçbir şey öğrenmeyen bir tahmin bile belli bir başarı alır.
Bir model ancak bu tabanı geçiyorsa gerçekten bir şey öğrenmiştir.

In [ ]:
mevsim = p["egitim_tum"].groupby(p["egitim_tum"].index.dayofyear)["hedef"].mean()
tahmin = {
    "Taban: doluluk değişmez": test["doluluk"].values,
    "Taban: mevsimsel ortalama": test["doluluk"].values + test.index.dayofyear.map(mevsim).fillna(0).values,
}
pt = bol(model_verisi(df))   # Doğrusal Regresyon ve XGBoost aynı günlerde
oz_tab = [c for c in pt["egitim_tum"].columns if c != "hedef"]
assert pt["test"].index.equals(test.index)
for ad, m in modeller().items():
    m.fit(pt["egitim_tum"][oz_tab], pt["egitim_tum"]["hedef"])
    tahmin[ad] = test["doluluk"].values + m.predict(pt["test"][oz_tab])
tahmin["LSTM (5 model ortalaması)"] = test["doluluk"].values + lstm_degisim

tablo = pd.DataFrame({ad: metrikler(gercek, t) for ad, t in tahmin.items()}).T.round(2)
tablo.columns = ["MAE (puan)", "RMSE (puan)", "R²"]
os.makedirs("../sonuclar", exist_ok=True)
tablo.to_csv("../sonuclar/karsilastirma_2023.csv")
tablo

In [ ]:
tarih = test.index + pd.Timedelta(days=UFUK)
plt.figure(figsize=(12, 4.5))
plt.plot(tarih, gercek, color="black", lw=2, label="Gerçek")
for ad in ["Taban: doluluk değişmez", "Doğrusal Regresyon", "XGBoost", "LSTM (5 model ortalaması)"]:
    plt.plot(tarih, tahmin[ad], lw=1.2, label=ad, ls="--" if ad.startswith("Taban") else "-")
plt.title(f"2023 testi: {UFUK} gün sonraki toplam baraj doluluğu"); plt.ylabel("Doluluk (%)"); plt.xlabel("Tarih")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig("../sonuclar/test_2023_tahminler.png", dpi=150); plt.show()

## 4. Yorum (rapor için taslak)
- Tablodaki sıralamayı yorumlayın: hangi model "doluluk değişmez" tabanını ne kadar geçiyor?
- 2023 eğitimde görülmemiş kadar kurak bir yıldı (doluluk %16'ya indi). Ağaç modelleri (XGBoost) eğitimde gördükleri
  aralığın dışına tahmin yapamaz; doğrusal model yapabilir. LSTM'in sonucu bu açıdan nerede duruyor?
- Aralık 2023'teki ani yükselişi hiçbir model yakalayamadı: o yükselişi 30 gün sonra yağacak yağmur yarattı ve
  modeller geleceğin yağışını bilmiyor. Bu, hava tahmini verisi eklemenin neden faydalı olabileceğini gösterir (gelecek çalışma).
- LSTM daha karmaşık bir model ama veri az (yaklaşık 4.000 gün). Karmaşık model her zaman daha iyi sonuç vermez;
  bu da raporda dürüstçe yazılabilecek bir bulgu.